# Solar Filament Segmentation 2026: re-tune the baseline model

Reuses the model trained by `kaggle_pipeline.ipynb` and redoes only the steps after training:
8-view test-time augmentation (flips and 90-degree rotations) and a wider post-processing search
on validation Panoptic Quality (low and high threshold, gap closing, hole filling, fragment
merging, minimum area). Then it predicts the test set and checks the submission file.

**Before running:** add two inputs, the competition data and the output of the notebook that
trained the model (*Add Input > Your Work > Notebooks*), turn on a GPU and turn Internet on.

In [ ]:
REPO_URL = "https://github.com/FarnooshMemari/solar-filament-segmentation.git"
REPO = "/tmp/solar-filament-segmentation"  # outside /kaggle/working, so it isn't saved as output
RUN = "/kaggle/working/runs/baseline"

import glob, os, shutil
if not os.path.exists(REPO):
    !git clone -q {REPO_URL} {REPO}
%cd {REPO}
!git log -1 --format="code version: %h %s"
!pip install -q pycocotools

found = glob.glob("/kaggle/input/**/runs/baseline/best.pt", recursive=True)
assert found, "Add the output of the training notebook as an input (Add Input > Your Work > Notebooks)."
shutil.copytree(os.path.dirname(found[0]), RUN, dirs_exist_ok=True)
print("model from:", os.path.dirname(found[0]))
print("run folder:", sorted(os.listdir(RUN)))
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Tune post-processing on validation PQ
Prints the best settings, the score of the previous settings with the same model, and the best
score possible with a single threshold, so the gain of each change is visible.

In [ ]:
!python scripts/tune.py --run {RUN} --tta 8 --workers 4

## 2. Predict the test set and check the file
`predict.py` uses the same test-time augmentation the settings were tuned with.

In [ ]:
!python scripts/predict.py --run {RUN} --out /kaggle/working/submission.csv
!python scripts/check_submission.py --csv /kaggle/working/submission.csv

## 3. Figures and package versions

In [ ]:
!python scripts/visualize.py --run {RUN} --n 4 --out /kaggle/working/figures
!pip freeze > /kaggle/working/requirements_frozen.txt
from IPython.display import Image, display
for path in sorted(glob.glob("/kaggle/working/figures/*.png"))[:2]:
    display(Image(path))

## 4. Submit

Submit `submission.csv` from the notebook's Output tab only if the check above printed
`submission=ok` and the validation PQ beat the previous settings.